# Loop de Convergencia - Clasificación Jerárquica con LLM en Ambos Niveles

## Objetivo
Procesar productos no clasificados en **batches pequeños (100 productos)** con **recálculo de centroids** después de cada batch exitoso.

## Estrategia de Clasificación Jerárquica (2 Niveles)

### 🟢 **Nivel 1: MACRO** (9 categorías generales)

```
1. Calcular similitud embedding: macro_sim

2. Decidir acción según macro_sim:

   ├─ Si macro_sim < 0.75 (MUY BAJO):
   │  └─ 🤖 Llamar ask_llm_macro()
   │     ├─ LLM confidence ≥ 0.7 → Usar macro del LLM, continuar a Nivel 2
   │     └─ LLM confidence < 0.7 → Review Queue
   │
   ├─ Si 0.75 ≤ macro_sim < 0.90 (MEDIO):
   │  └─ Usar macro del embedding, continuar a Nivel 2
   │     (LLM se usará en Nivel 2 para validar categoría)
   │
   └─ Si macro_sim ≥ 0.90 (ALTO):
      └─ Usar macro del embedding, continuar a Nivel 2
         (Alta confianza, no necesita LLM en Nivel 2)
```

### 🔵 **Nivel 2: CATEGORIA** (dentro del macro seleccionado)

```
1. Calcular similitud embedding: cat_sim (dentro de categorías del macro)

2. Decidir acción según macro_sim original:

   ├─ Si macro_sim ≥ 0.90 Y cat_sim ≥ 0.75:
   │  └─ ✅ AUTO-ASIGNAR (alta confianza en ambos niveles)
   │
   ├─ Si 0.75 ≤ macro_sim < 0.90 O macro decisión por LLM:
   │  └─ 🤖 Llamar ask_llm() para validar categoría
   │     ├─ LLM confidence ≥ 0.7 → ✅ AUTO-ASIGNAR
   │     └─ LLM confidence < 0.7 → Review Queue
   │
   └─ Si cat_sim < 0.75:
      └─ Review Queue (baja confianza en categoría)
```

---

## Flujo de Iteración
1. **Por cada iteración** (30 máximo):
   - Tomar 100 productos random no clasificados
   - Clasificar con enfoque jerárquico (descripto arriba)
   - Si hay productos auto-asignados:
     - 💾 Guardar en `gold_productos_categorias`
     - 🔧 **Ejecutar notebook 05** para recalcular centroids
     - 🔄 Recargar centroids en memoria
   - Si NO hay productos auto-asignados:
     - ➡️ Continuar (centroids no cambian)

2. **Tracking de convergencia:**
   - % de requeue por iteración
   - Productos auto-asignados acumulados
   - **LLM Nivel 1** (Macro): llamadas, éxitos, tasa de éxito
   - **LLM Nivel 2** (Categoría): llamadas, éxitos, tasa de éxito
   - Ver si mejora con el tiempo

---

## Hipótesis de Convergencia

A medida que agregamos productos, los centroids se refinan y **deberían capturar mejor la distribución de productos "difíciles"** → el % de requeue debería **disminuir**.

---

## Mejoras del Prompt LLM (vs Notebook 07)

### 🤖 **LLM Macro** (Nivel 1 - ask_llm_macro)
✅ **9 opciones**: Solo macros (Almacén, Bebidas, Frescos, etc.)  
✅ **Descripciones**: Incluye descripción de cada macro  
✅ **Conservador**: Puede responder "ninguna" si no está seguro  
✅ **Validado**: Solo auto-asigna con confidence ≥ 0.7  

### 🤖 **LLM Categoría** (Nivel 2 - ask_llm)
✅ **Neutro**: No está sesgado hacia ninguna categoría específica  
✅ **Completo**: Pasa las 139 categorías (no solo 60)  
✅ **Contextual**: Top 3 candidatos con ejemplos reales de productos  
✅ **Conservador**: Permite decir "ninguna" si no está seguro  
✅ **Validado**: Solo auto-asigna con confidence ≥ 0.7  

---

## Ventajas del Enfoque de 2 Niveles con LLM

1. **Eficiencia**: Embeddings primero (rápido), LLM solo cuando necesario
2. **Precisión**: LLM ayuda en casos ambiguos donde embeddings fallan
3. **Escalabilidad**: LLM se llama solo en ~20-40% de casos (no en todos)
4. **Jerárquico**: Decidir macro primero reduce complejidad en nivel 2
5. **Convergencia**: Centroids mejoran con cada batch, reduciendo necesidad de LLM

---

In [0]:
%pip install -q sentence-transformers
dbutils.library.restartPython()

In [0]:
import json
import numpy as np
import pandas as pd
from datetime import datetime
from sentence_transformers import SentenceTransformer
import time

# Configuration
MACRO_AUTO = 0.90      # Threshold para auto-asignar MACRO (alto)
CATEGORY_AUTO = 0.75   # Threshold para auto-asignar CATEGORIA (alto)
BATCH_SIZE = 100       # Productos por iteración
MAX_ITERATIONS = 30    # Máximo de iteraciones (3,000 productos total)

print(f'🔧 Configuración:')
print(f'   MACRO_AUTO:     {MACRO_AUTO}')
print(f'   CATEGORY_AUTO:  {CATEGORY_AUTO}')
print(f'   BATCH_SIZE:     {BATCH_SIZE}')
print(f'   MAX_ITERATIONS: {MAX_ITERATIONS}')
print(f'   Total productos a procesar: {BATCH_SIZE * MAX_ITERATIONS:,}')
print()

In [0]:
# LLM Configuration
USE_LLM = True                    # Habilitar LLM para validación
LLM_MACRO_MIN = 0.75              # Similitud macro mínima para llamar LLM
LLM_MACRO_MAX = 0.90              # Similitud macro máxima para llamar LLM (por encima se auto-asigna)
LLM_TOP_CANDIDATES = 3            # Número de categorías candidatas a mostrar al LLM

print(f'🤖 Configuración LLM:')
print(f'   USE_LLM:          {USE_LLM}')
print(f'   LLM_MACRO_MIN:    {LLM_MACRO_MIN} (mínimo para llamar LLM)')
print(f'   LLM_MACRO_MAX:    {LLM_MACRO_MAX} (máximo - por encima auto-asigna)')
print(f'   LLM_TOP_CANDIDATES: {LLM_TOP_CANDIDATES}')
print()

In [0]:
def ask_llm(producto, top_categories, all_categories_dict):
    """
    Pregunta al LLM cuál categoría es la mejor para el producto.
    
    PROMPT MEJORADO:
    - Neutro: no está sesgado hacia ninguna categoría
    - Completo: pasa TODAS las 139 categorías disponibles
    - Top candidatos con ejemplos: ayuda al LLM a decidir
    - Permite decir "ninguna" si no está seguro
    
    Args:
        producto: str - Nombre del producto
        top_categories: list of tuples - [(cat_name, similarity, examples), ...]
        all_categories_dict: dict - {cat_name: [examples], ...} - TODAS las categorías
    
    Returns:
        tuple: (categoria_sugerida, confidence_score) o (None, 0.0)
    """
    import json
    
    # Build prompt con top candidatos
    top_candidates_text = "\n".join([
        f"  {i+1}. {cat} (similitud: {sim:.3f})\n     Ejemplos: {', '.join(examples[:3])}"
        for i, (cat, sim, examples) in enumerate(top_categories)
    ])
    
    # Lista completa de categorías (solo nombres)
    all_categories_list = sorted(all_categories_dict.keys())
    all_categories_text = ", ".join(all_categories_list)
    
    prompt = f"""Eres un experto en clasificación de productos de supermercado argentino.

Producto a clasificar: "{producto}"

## Top {len(top_categories)} Candidatos (por similitud de embeddings):
{top_candidates_text}

## Categorías Disponibles ({len(all_categories_list)} total):
{all_categories_text}

## Tarea:
Analiza el producto y determina a qué categoría pertenece. Considera:
1. Los candidatos sugeridos por similitud semántica
2. Todas las categorías disponibles
3. El contexto del producto (ingredientes, uso, marca)

Si NINGUNA categoría es apropiada, responde "ninguna".

## Respuesta (JSON):
Responde SOLO con JSON válido:
{{
  "categoria": "nombre de la categoría" o "ninguna",
  "confianza": 0.0-1.0,
  "razon": "breve explicación"
}}
"""
    
    try:
        # Call LLM via AI Gateway
        response = spark.sql(f"""
            SELECT ai_query(
                'superapp_endpoint',
                {json.dumps(prompt)}
            ) as respuesta
        """).collect()[0]['respuesta']
        
        # Parse JSON response
        result = json.loads(response)
        categoria = result.get('categoria', 'ninguna')
        confianza = float(result.get('confianza', 0.0))
        razon = result.get('razon', '')
        
        if categoria.lower() == 'ninguna' or categoria not in all_categories_dict:
            return None, 0.0
        
        return categoria, confianza
        
    except Exception as e:
        print(f"   ⚠️ Error LLM: {str(e)[:100]}")
        return None, 0.0

print('✅ Función ask_llm() definida (prompt neutro y completo)')

In [0]:
def ask_llm_macro(producto, macro_names):
    """
    Pregunta al LLM a cuál MACRO pertenece el producto (nivel 1).
    Útil cuando el embedding tiene baja similitud (< 0.75).
    
    Args:
        producto: str - Nombre del producto
        macro_names: list - Lista de nombres de macros (9 opciones)
    
    Returns:
        tuple: (macro_nombre, confidence_score) o (None, 0.0)
    """
    import json
    
    macros_text = ", ".join(macro_names)
    
    prompt = f"""Eres un experto en clasificación de productos de supermercado argentino.

Producto: "{producto}"

## Macro-Categorías Disponibles ({len(macro_names)} opciones):
{macros_text}

## Descripción de Macros:
- **Almacén**: Productos secos, envasados, conservas (fideos, arroz, latas, salsas, aceites, condimentos)
- **Bebidas**: Todas las bebidas (gaseosas, jugos, aguas, cervezas, vinos, energizantes)
- **Frescos**: Productos refrigerados perecederos (lácteos, fiambres, quesos, yogures)
- **Congelados**: Productos congelados (helados, vegetales congelados, comidas congeladas)
- **Limpieza**: Productos de limpieza del hogar (detergentes, lavandina, limpiadores)
- **Perfumería**: Cuidado personal e higiene (shampoo, jabón, desodorante, cremas, pañales)
- **Electro**: Electrodomésticos y electrónica
- **Textil**: Ropa, calzado, accesorios
- **Hogar**: Bazar, decoración, utensilios de cocina

## Tarea:
Determina a qué macro-categoría pertenece el producto. Si NO estás seguro, responde "ninguna".

## Respuesta (JSON):
Responde SOLO con JSON válido:
{{
  "macro": "nombre de la macro" o "ninguna",
  "confianza": 0.0-1.0,
  "razon": "breve explicación"
}}
"""
    
    try:
        response = spark.sql(f"""
            SELECT ai_query(
                'superapp_endpoint',
                {json.dumps(prompt)}
            ) as respuesta
        """).collect()[0]['respuesta']
        
        result = json.loads(response)
        macro = result.get('macro', 'ninguna')
        confianza = float(result.get('confianza', 0.0))
        
        if macro.lower() == 'ninguna' or macro not in macro_names:
            return None, 0.0
        
        return macro, confianza
        
    except Exception as e:
        print(f"   ⚠️ Error LLM macro: {str(e)[:100]}")
        return None, 0.0

print('✅ Función ask_llm_macro() definida (decisión de macro para productos difíciles)')

In [0]:
def load_centroids():
    """
    Carga centroids de MACRO y CATEGORIA desde las tablas gold.
    Retorna: (macro_matrix, macro_names, macro_id_map, categories_by_macro)
    """
    # Load MACRO centroids (9 macros)
    macro_pd = spark.table('workspace.superapp.gold_macro_centroids').toPandas()
    macro_pd['centroid'] = macro_pd['centroid_json'].apply(json.loads).apply(np.array)
    
    macro_matrix = np.array(macro_pd['centroid'].tolist())  # (9, 384)
    macro_names = macro_pd['nombre'].tolist()
    macro_id_map = dict(zip(macro_pd['nombre'], macro_pd['id_macro']))
    
    # Load CATEGORY centroids (139 categories) with macro mapping
    cat_pd = spark.sql("""
        SELECT 
            cc.id_categoria,
            cc.nombre,
            cc.centroid_json,
            c.id_macro
        FROM workspace.superapp.gold_category_centroids cc
        JOIN workspace.superapp.gold_categorias c ON cc.id_categoria = c.id_categoria
        WHERE c.id_macro IS NOT NULL
    """).toPandas()
    
    cat_pd['centroid'] = cat_pd['centroid_json'].apply(json.loads).apply(np.array)
    
    # Group categories by macro
    categories_by_macro = {}
    for macro_id in macro_id_map.values():
        macro_cats = cat_pd[cat_pd['id_macro'] == macro_id]
        if len(macro_cats) > 0:
            categories_by_macro[macro_id] = {
                'names': macro_cats['nombre'].tolist(),
                'ids': macro_cats['id_categoria'].tolist(),
                'matrix': np.array(macro_cats['centroid'].tolist())
            }
    
    return macro_matrix, macro_names, macro_id_map, categories_by_macro

print('✅ Función load_centroids() definida')

In [0]:
def cosine_sim_batch(embeddings, centroid_matrix):
    """Calcula similitud coseno entre embeddings y centroids"""
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    normed = embeddings / (norms + 1e-10)
    return normed @ centroid_matrix.T

def classify_batch(batch_df, model, macro_matrix, macro_names, macro_id_map, categories_by_macro):
    """
    Clasifica un batch de productos usando el enfoque jerárquico + LLM.
    
    Flujo:
    - macro_sim < 0.75: LLM decide MACRO + CATEGORIA (full LLM path)
    - 0.75 ≤ macro_sim < 0.90: Embedding MACRO + LLM valida CATEGORIA
    - macro_sim ≥ 0.90: Embedding MACRO + CATEGORIA (alta confianza)
    
    Retorna: (auto_assigned, review_queue, stats)
    """
    auto_assigned = []
    review_queue = []
    stats = {
        'level1_success': 0, 
        'level2_success': 0, 
        'llm_called': 0, 
        'llm_success': 0,
        'llm_full_path': 0,  # LLM decidió MACRO + CATEGORIA
        'llm_category_only': 0  # LLM solo validó CATEGORIA
    }
    
    # Load all categories with examples (for LLM)
    all_categories_dict = {}
    if USE_LLM:
        cat_examples = spark.sql("""
            SELECT c.nombre, sp.producto
            FROM workspace.superapp.gold_productos_categorias pc
            JOIN workspace.superapp.silver_prices sp ON pc.id_producto = sp.id_producto
            JOIN workspace.superapp.gold_categorias c ON pc.id_categoria = c.id_categoria
            WHERE pc.metodo != 'manual'
            DISTRIBUTE BY c.nombre
            SORT BY c.nombre, RAND()
        """).toPandas()
        
        for cat_name, group in cat_examples.groupby('nombre'):
            all_categories_dict[cat_name] = group['producto'].head(5).tolist()
    
    # Generar embeddings
    embs = model.encode(batch_df['producto'].tolist(), batch_size=128, show_progress_bar=False)
    
    # LEVEL 1: Classify by MACRO
    macro_sims = cosine_sim_batch(embs, macro_matrix)
    macro_top_idx = macro_sims.argmax(axis=1)
    macro_top_scores = macro_sims.max(axis=1)
    
    for i, (_, row) in enumerate(batch_df.iterrows()):
        macro_score = float(macro_top_scores[i])
        macro_idx = macro_top_idx[i]
        macro_name_embedding = macro_names[macro_idx]
        
        # ========================================
        # PATH 1: FULL LLM (macro_sim < 0.75)
        # ========================================
        if USE_LLM and macro_score < LLM_MACRO_MIN:
            stats['llm_called'] += 1
            stats['llm_full_path'] += 1
            
            # Step 1: LLM decide MACRO
            llm_macro, llm_macro_conf = ask_llm_macro(row['producto'], macro_names)
            
            if not llm_macro or llm_macro_conf < 0.7:
                # LLM uncertain about macro - review queue
                review_queue.append({
                    'id_producto': row['id_producto'],
                    'producto': row['producto'],
                    'razon': 'llm_macro_low_confidence',
                    'top_macro': macro_name_embedding,
                    'similitud_macro': macro_score,
                    'top_categoria': None,
                    'similitud': None,
                    'llm_sugerencia': f'macro_llm={llm_macro}({llm_macro_conf:.2f})' if llm_macro else 'ninguna',
                    'estado': 'pendiente',
                    'fecha_creacion': datetime.now()
                })
                continue
            
            # Step 2: LLM decide CATEGORIA within macro
            macro_id = macro_id_map[llm_macro]
            
            if macro_id not in categories_by_macro:
                review_queue.append({
                    'id_producto': row['id_producto'],
                    'producto': row['producto'],
                    'razon': 'no_categories_in_macro',
                    'top_macro': llm_macro,
                    'similitud_macro': llm_macro_conf,
                    'top_categoria': None,
                    'similitud': None,
                    'llm_sugerencia': f'macro_llm={llm_macro}',
                    'estado': 'pendiente',
                    'fecha_creacion': datetime.now()
                })
                continue
            
            macro_data = categories_by_macro[macro_id]
            cat_names = macro_data['names']
            cat_ids = macro_data['ids']
            
            # Get top N candidates from within the LLM-selected macro
            cat_matrix = macro_data['matrix']
            cat_sims = cosine_sim_batch(embs[i:i+1], cat_matrix)[0]
            top_indices = cat_sims.argsort()[-LLM_TOP_CANDIDATES:][::-1]
            
            top_candidates = []
            for idx in top_indices:
                cat_candidate = cat_names[idx]
                sim = float(cat_sims[idx])
                examples = all_categories_dict.get(cat_candidate, [])
                top_candidates.append((cat_candidate, sim, examples))
            
            # Ask LLM for category
            llm_cat, llm_cat_conf = ask_llm(row['producto'], top_candidates, all_categories_dict)
            
            if llm_cat and llm_cat_conf >= 0.7:
                # Find category ID
                llm_cat_id = None
                for idx, name in enumerate(cat_names):
                    if name == llm_cat:
                        llm_cat_id = cat_ids[idx]
                        break
                
                if llm_cat_id:
                    auto_assigned.append({
                        'id_producto': row['id_producto'],
                        'id_categoria': int(llm_cat_id),
                        'id_subcategoria': None,
                        'metodo': 'embedding_hierarchical_llm_full_convergence',
                        'confianza': llm_cat_conf,
                        'fecha_asignacion': datetime.now(),
                        'usuario_asignacion': 'system',
                        'notas': f'llm_macro={llm_macro}({llm_macro_conf:.3f})|llm_cat={llm_cat}({llm_cat_conf:.3f})'
                    })
                    stats['llm_success'] += 1
                else:
                    review_queue.append({
                        'id_producto': row['id_producto'],
                        'producto': row['producto'],
                        'razon': 'llm_category_not_found',
                        'top_macro': llm_macro,
                        'similitud_macro': llm_macro_conf,
                        'top_categoria': llm_cat,
                        'similitud': llm_cat_conf,
                        'llm_sugerencia': f'{llm_macro}>{llm_cat}',
                        'estado': 'pendiente',
                        'fecha_creacion': datetime.now()
                    })
            else:
                review_queue.append({
                    'id_producto': row['id_producto'],
                    'producto': row['producto'],
                    'razon': 'llm_category_low_confidence',
                    'top_macro': llm_macro,
                    'similitud_macro': llm_macro_conf,
                    'top_categoria': llm_cat if llm_cat else None,
                    'similitud': llm_cat_conf if llm_cat else None,
                    'llm_sugerencia': f'{llm_macro}>{llm_cat}' if llm_cat else f'{llm_macro}>ninguna',
                    'estado': 'pendiente',
                    'fecha_creacion': datetime.now()
                })
            
            continue
        
        # ========================================
        # PATH 2 & 3: Embedding decides MACRO
        # ========================================
        
        macro_id = macro_id_map[macro_name_embedding]
        stats['level1_success'] += 1
        
        # Check if macro has categories
        if macro_id not in categories_by_macro:
            review_queue.append({
                'id_producto': row['id_producto'],
                'producto': row['producto'],
                'razon': 'no_categories_in_macro',
                'top_macro': macro_name_embedding,
                'similitud_macro': macro_score,
                'top_categoria': None,
                'similitud': None,
                'llm_sugerencia': None,
                'estado': 'pendiente',
                'fecha_creacion': datetime.now()
            })
            continue
        
        macro_data = categories_by_macro[macro_id]
        cat_matrix = macro_data['matrix']
        cat_names = macro_data['names']
        cat_ids = macro_data['ids']
        
        cat_sims = cosine_sim_batch(embs[i:i+1], cat_matrix)[0]
        cat_top_idx = cat_sims.argmax()
        cat_top_score = float(cat_sims[cat_top_idx])
        cat_name = cat_names[cat_top_idx]
        cat_id = cat_ids[cat_top_idx]
        
        # ========================================
        # PATH 3: HIGH confidence (macro ≥ 0.90, cat ≥ 0.75)
        # ========================================
        if macro_score >= LLM_MACRO_MAX and cat_top_score >= CATEGORY_AUTO:
            auto_assigned.append({
                'id_producto': row['id_producto'],
                'id_categoria': int(cat_id),
                'id_subcategoria': None,
                'metodo': 'embedding_hierarchical_convergence',
                'confianza': cat_top_score,
                'fecha_asignacion': datetime.now(),
                'usuario_asignacion': 'system',
                'notas': f'macro={macro_name_embedding}({macro_score:.3f})|cat={cat_name}({cat_top_score:.3f})'
            })
            stats['level2_success'] += 1
            
        # ========================================
        # PATH 2: MEDIUM macro confidence (0.75 ≤ macro < 0.90) - LLM validates category
        # ========================================
        elif USE_LLM and LLM_MACRO_MIN <= macro_score < LLM_MACRO_MAX:
            stats['llm_called'] += 1
            stats['llm_category_only'] += 1
            
            # Get top N candidates with examples
            top_indices = cat_sims.argsort()[-LLM_TOP_CANDIDATES:][::-1]
            top_candidates = []
            for idx in top_indices:
                cat_candidate = cat_names[idx]
                sim = float(cat_sims[idx])
                examples = all_categories_dict.get(cat_candidate, [])
                top_candidates.append((cat_candidate, sim, examples))
            
            # Ask LLM
            llm_cat, llm_conf = ask_llm(row['producto'], top_candidates, all_categories_dict)
            
            if llm_cat and llm_conf >= 0.7:
                # LLM confirmed - auto-assign
                llm_cat_id = None
                for idx, name in enumerate(cat_names):
                    if name == llm_cat:
                        llm_cat_id = cat_ids[idx]
                        break
                
                if llm_cat_id:
                    auto_assigned.append({
                        'id_producto': row['id_producto'],
                        'id_categoria': int(llm_cat_id),
                        'id_subcategoria': None,
                        'metodo': 'embedding_hierarchical_llm_convergence',
                        'confianza': llm_conf,
                        'fecha_asignacion': datetime.now(),
                        'usuario_asignacion': 'system',
                        'notas': f'macro={macro_name_embedding}({macro_score:.3f})|llm={llm_cat}({llm_conf:.3f})'
                    })
                    stats['llm_success'] += 1
                else:
                    review_queue.append({
                        'id_producto': row['id_producto'],
                        'producto': row['producto'],
                        'razon': 'llm_category_not_in_macro',
                        'top_macro': macro_name_embedding,
                        'similitud_macro': macro_score,
                        'top_categoria': cat_name,
                        'similitud': cat_top_score,
                        'llm_sugerencia': llm_cat,
                        'estado': 'pendiente',
                        'fecha_creacion': datetime.now()
                    })
            else:
                review_queue.append({
                    'id_producto': row['id_producto'],
                    'producto': row['producto'],
                    'razon': 'llm_low_confidence',
                    'top_macro': macro_name_embedding,
                    'similitud_macro': macro_score,
                    'top_categoria': cat_name,
                    'similitud': cat_top_score,
                    'llm_sugerencia': llm_cat if llm_cat else 'ninguna',
                    'estado': 'pendiente',
                    'fecha_creacion': datetime.now()
                })
        else:
            # LOW category similarity - review queue
            review_queue.append({
                'id_producto': row['id_producto'],
                'producto': row['producto'],
                'razon': 'low_category_similarity',
                'top_macro': macro_name_embedding,
                'similitud_macro': macro_score,
                'top_categoria': cat_name,
                'similitud': cat_top_score,
                'llm_sugerencia': None,
                'estado': 'pendiente',
                'fecha_creacion': datetime.now()
            })
    
    return auto_assigned, review_queue, stats

print('✅ Función classify_batch() definida (con Full LLM Path para productos difíciles)')

In [0]:
def rebuild_centroids():
    """
    Ejecuta el notebook 05 para recalcular centroids con los nuevos productos clasificados.
    Retorna True si exitoso, False si falla.
    """
    try:
        print('   🔧 Ejecutando notebook 05 (build_category_embeddings)...')
        
        # Ejecutar notebook 05
        result = dbutils.notebook.run(
            '/Users/franciscobarber@gmail.com/superapp/SuperApp/gold_setup/05_build_category_embeddings',
            timeout_seconds=600  # 10 minutos timeout
        )
        
        print('   ✅ Centroids recalculados exitosamente')
        return True
        
    except Exception as e:
        print(f'   ❌ Error al ejecutar notebook 05: {str(e)[:100]}')
        return False

print('✅ Función rebuild_centroids() definida')

In [0]:
# Load model
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print('✅ Modelo cargado')

# Load initial centroids
print('\n🔧 Cargando centroids iniciales...')
macro_matrix, macro_names, macro_id_map, categories_by_macro = load_centroids()
print(f'✅ Centroids cargados: {len(macro_names)} macros, {sum(len(v["names"]) for v in categories_by_macro.values())} categorías')

# Track results across iterations
iteration_results = []
total_auto_assigned = 0
total_products_processed = 0
centroids_rebuilt_count = 0
total_llm_called = 0
total_llm_success = 0
total_llm_full_path = 0
total_llm_category_only = 0

print('\n' + '='*80)
print('🚀 INICIANDO LOOP DE CONVERGENCIA' + (' CON LLM (FULL PATH)' if USE_LLM else ''))
print('='*80)

for iteration in range(1, MAX_ITERATIONS + 1):
    print(f'\n--- Iteración {iteration}/{MAX_ITERATIONS} ---')
    
    # Get remaining unclassified products
    unclassified_current = spark.sql("""
        SELECT DISTINCT sp.id_producto, sp.producto
        FROM workspace.superapp.silver_prices sp
        LEFT JOIN workspace.superapp.gold_productos_categorias pc
            ON sp.id_producto = pc.id_producto
        WHERE (pc.id_producto IS NULL OR pc.id_categoria IS NULL)
          AND sp.producto IS NOT NULL
    """).toPandas()
    
    remaining = len(unclassified_current)
    print(f'Productos no clasificados: {remaining:,}')
    
    if remaining == 0:
        print('✅ No hay más productos para clasificar')
        break
    
    # Random sample
    if remaining >= BATCH_SIZE:
        sample = unclassified_current.sample(n=BATCH_SIZE, random_state=42+iteration)
    else:
        sample = unclassified_current
        print(f'⚠️  Solo {len(sample):,} productos restantes')
    
    # Classify batch
    auto_assigned, review_queue, stats = classify_batch(
        sample, model, macro_matrix, macro_names, macro_id_map, categories_by_macro
    )
    
    # Stats
    num_auto = len(auto_assigned)
    num_review = len(review_queue)
    pct_auto = (num_auto / len(sample)) * 100
    pct_review = (num_review / len(sample)) * 100
    
    print(f'✅ Auto-asignados: {num_auto} ({pct_auto:.1f}%)')
    print(f'⚠️  Review queue:   {num_review} ({pct_review:.1f}%)')
    print(f'   Level 1 (macro):    {stats["level1_success"]} pasaron threshold')
    print(f'   Level 2 (category): {stats["level2_success"]} auto-asignados')
    
    if USE_LLM:
        llm_success_rate = (stats['llm_success'] / stats['llm_called'] * 100) if stats['llm_called'] > 0 else 0
        print(f'   🤖 LLM llamadas:      {stats["llm_called"]}')
        print(f'   🤖 LLM exitosas:     {stats["llm_success"]} ({llm_success_rate:.1f}%)')
        print(f'      └─ Full path (macro+cat): {stats["llm_full_path"]}')
        print(f'      └─ Category only: {stats["llm_category_only"]}')
    
    # Save auto-assigned products
    if auto_assigned:
        from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, TimestampType
        schema_auto = StructType([
            StructField('id_producto', StringType(), False),
            StructField('id_categoria', LongType(), True),
            StructField('id_subcategoria', LongType(), True),
            StructField('metodo', StringType(), True),
            StructField('confianza', DoubleType(), True),
            StructField('fecha_asignacion', TimestampType(), True),
            StructField('usuario_asignacion', StringType(), True),
            StructField('notas', StringType(), True),
        ])
        
        (spark.createDataFrame(auto_assigned, schema=schema_auto)
              .write.mode('append').option('mergeSchema', 'true')
              .saveAsTable('workspace.superapp.gold_productos_categorias'))
        
        print(f'💾 Guardados en gold_productos_categorias')
        
        # Rebuild centroids
        success = rebuild_centroids()
        if success:
            # Reload centroids
            print('   🔄 Recargando centroids...')
            macro_matrix, macro_names, macro_id_map, categories_by_macro = load_centroids()
            centroids_rebuilt_count += 1
            print(f'   ✅ Centroids recargados (rebuild #{centroids_rebuilt_count})')
        else:
            print('   ⚠️  Continuando con centroids existentes')
    
    # Track results
    iteration_results.append({
        'iteration': iteration,
        'remaining_products': remaining,
        'batch_size': len(sample),
        'auto_assigned': num_auto,
        'review_queue': num_review,
        'pct_auto': pct_auto,
        'pct_review': pct_review,
        'level1_success': stats['level1_success'],
        'level2_success': stats['level2_success'],
        'llm_called': stats.get('llm_called', 0),
        'llm_success': stats.get('llm_success', 0),
        'llm_full_path': stats.get('llm_full_path', 0),
        'llm_category_only': stats.get('llm_category_only', 0),
        'centroids_rebuilt': success if auto_assigned else False
    })
    
    total_auto_assigned += num_auto
    total_products_processed += len(sample)
    total_llm_called += stats.get('llm_called', 0)
    total_llm_success += stats.get('llm_success', 0)
    total_llm_full_path += stats.get('llm_full_path', 0)
    total_llm_category_only += stats.get('llm_category_only', 0)
    
    # Small delay to avoid overloading
    time.sleep(1)

print('\n' + '='*80)
print('📊 RESUMEN FINAL')
print('='*80)
print(f'Total iteraciones:         {len(iteration_results)}')
print(f'Total productos procesados: {total_products_processed:,}')
print(f'Total auto-asignados:      {total_auto_assigned:,} ({total_auto_assigned/total_products_processed*100:.1f}%)')
print(f'Centroids reconstruidos:   {centroids_rebuilt_count} veces')
if USE_LLM:
    llm_total_rate = (total_llm_success / total_llm_called * 100) if total_llm_called > 0 else 0
    print(f'\n🤖 LLM Performance:')
    print(f'   Total llamadas:     {total_llm_called:,}')
    print(f'   Total exitosas:     {total_llm_success:,} ({llm_total_rate:.1f}%)')
    print(f'   Full path (macro+cat): {total_llm_full_path:,}')
    print(f'   Category only:      {total_llm_category_only:,}')
print('='*80)

In [0]:
# Create DataFrame with results
import pandas as pd
import matplotlib.pyplot as plt

results_df = pd.DataFrame(iteration_results)

# Calculate total LLM metrics
if USE_LLM:
    results_df['llm_total_called'] = results_df['llm_macro_called'] + results_df['llm_category_called']
    results_df['llm_total_success'] = results_df['llm_macro_success'] + results_df['llm_category_success']

print('\n📊 RESULTADOS POR ITERACIÓN')
print('='*120)
if USE_LLM:
    display_cols = ['iteration', 'batch_size', 'auto_assigned', 'pct_auto', 'pct_review', 
                    'llm_macro_called', 'llm_macro_success', 'llm_category_called', 'llm_category_success', 
                    'centroids_rebuilt']
    print(results_df[display_cols].to_string(index=False))
else:
    print(results_df[['iteration', 'batch_size', 'auto_assigned', 'pct_auto', 'pct_review', 'centroids_rebuilt']].to_string(index=False))

# Plot convergence
if len(results_df) > 1:
    total_llm_calls = results_df['llm_total_called'].sum() if USE_LLM else 0
    
    if USE_LLM and total_llm_calls > 0:
        # 6 subplots with LLM metrics
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    else:
        # 4 subplots without LLM
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    axes_flat = axes.flatten() if isinstance(axes, np.ndarray) else [axes]
    
    # 1. % Auto-asignado por iteración
    axes_flat[0].plot(results_df['iteration'], results_df['pct_auto'], marker='o', color='green')
    axes_flat[0].axhline(y=0, color='red', linestyle='--', alpha=0.3)
    axes_flat[0].set_xlabel('Iteración')
    axes_flat[0].set_ylabel('% Auto-asignado')
    axes_flat[0].set_title('Convergencia: % Auto-asignado por Iteración')
    axes_flat[0].grid(True, alpha=0.3)
    
    # 2. Auto-asignados acumulados
    results_df['cumulative_auto'] = results_df['auto_assigned'].cumsum()
    axes_flat[1].plot(results_df['iteration'], results_df['cumulative_auto'], marker='o', color='blue')
    axes_flat[1].set_xlabel('Iteración')
    axes_flat[1].set_ylabel('Productos Auto-asignados (acumulado)')
    axes_flat[1].set_title('Productos Auto-asignados Acumulados')
    axes_flat[1].grid(True, alpha=0.3)
    
    # 3. Level 1 vs Level 2 success + LLM
    if len(axes_flat) > 2:
        axes_flat[2].plot(results_df['iteration'], results_df['level1_success'], marker='o', label='Level 1 (Macro)', color='orange')
        axes_flat[2].plot(results_df['iteration'], results_df['level2_success'], marker='s', label='Level 2 (Embedding)', color='purple')
        if USE_LLM and total_llm_calls > 0:
            axes_flat[2].plot(results_df['iteration'], results_df['llm_macro_success'], marker='^', label='LLM Macro', color='cyan')
            axes_flat[2].plot(results_df['iteration'], results_df['llm_category_success'], marker='d', label='LLM Categoría', color='magenta')
        axes_flat[2].set_xlabel('Iteración')
        axes_flat[2].set_ylabel('Productos')
        axes_flat[2].set_title('Éxito por Nivel y Método')
        axes_flat[2].legend()
        axes_flat[2].grid(True, alpha=0.3)
    
    # 4. Productos no clasificados restantes
    if len(axes_flat) > 3:
        axes_flat[3].plot(results_df['iteration'], results_df['remaining_products'], marker='o', color='red')
        axes_flat[3].set_xlabel('Iteración')
        axes_flat[3].set_ylabel('Productos Restantes')
        axes_flat[3].set_title('Productos No Clasificados (restantes)')
        axes_flat[3].grid(True, alpha=0.3)
    
    # 5. LLM: Llamadas por nivel (if LLM enabled)
    if USE_LLM and len(axes_flat) > 4 and total_llm_calls > 0:
        x = results_df['iteration']
        width = 0.35
        
        axes_flat[4].bar(x - width/2, results_df['llm_macro_called'], width, alpha=0.7, label='Macro (Nivel 1)', color='lightblue')
        axes_flat[4].bar(x + width/2, results_df['llm_category_called'], width, alpha=0.7, label='Categoría (Nivel 2)', color='lightcoral')
        
        # Overlay success bars
        axes_flat[4].bar(x - width/2, results_df['llm_macro_success'], width, alpha=0.9, label='Macro exitosas', color='darkblue')
        axes_flat[4].bar(x + width/2, results_df['llm_category_success'], width, alpha=0.9, label='Categoría exitosas', color='darkred')
        
        axes_flat[4].set_xlabel('Iteración')
        axes_flat[4].set_ylabel('Cantidad')
        axes_flat[4].set_title('LLM: Llamadas y Éxitos por Nivel')
        axes_flat[4].legend()
        axes_flat[4].grid(True, alpha=0.3)
        
        # 6. LLM: Tasa de éxito (%) por nivel
        results_df['llm_macro_success_rate'] = (results_df['llm_macro_success'] / results_df['llm_macro_called'] * 100).fillna(0)
        results_df['llm_category_success_rate'] = (results_df['llm_category_success'] / results_df['llm_category_called'] * 100).fillna(0)
        results_df['llm_total_success_rate'] = (results_df['llm_total_success'] / results_df['llm_total_called'] * 100).fillna(0)
        
        axes_flat[5].plot(results_df['iteration'], results_df['llm_macro_success_rate'], marker='o', label='Nivel 1 (Macro)', color='blue')
        axes_flat[5].plot(results_df['iteration'], results_df['llm_category_success_rate'], marker='s', label='Nivel 2 (Categoría)', color='red')
        axes_flat[5].plot(results_df['iteration'], results_df['llm_total_success_rate'], marker='^', label='Total', color='purple', linewidth=2)
        axes_flat[5].axhline(y=50, color='orange', linestyle='--', alpha=0.3, label='50%')
        axes_flat[5].set_xlabel('Iteración')
        axes_flat[5].set_ylabel('% Éxito')
        axes_flat[5].set_title('Tasa de Éxito del LLM por Nivel')
        axes_flat[5].legend()
        axes_flat[5].grid(True, alpha=0.3)
    
    plt.tight_layout()
    display(plt.show())
    
    # Analysis
    print('\n🔍 ANÁLISIS DE CONVERGENCIA')
    print('='*120)
    
    if results_df['pct_auto'].max() > 0:
        print(f'✅ Mejor iteración: #{results_df["pct_auto"].idxmax() + 1} con {results_df["pct_auto"].max():.1f}% auto-asignado')
        
        # Check if there's improvement over time
        first_half = results_df.iloc[:len(results_df)//2]['pct_auto'].mean()
        second_half = results_df.iloc[len(results_df)//2:]['pct_auto'].mean()
        
        if second_half > first_half:
            print(f'✅ CONVERGENCIA POSITIVA: Segunda mitad ({second_half:.1f}%) > Primera mitad ({first_half:.1f}%)')
        elif second_half < first_half:
            print(f'⚠️  CONVERGENCIA NEGATIVA: Segunda mitad ({second_half:.1f}%) < Primera mitad ({first_half:.1f}%)')
        else:
            print(f'➡️  SIN CAMBIO: Ambas mitades con {first_half:.1f}% promedio')
        
        # LLM analysis by level
        if USE_LLM and total_llm_calls > 0:
            print(f'\n🤖 LLM Performance por Nivel:')
            
            # Macro (Nivel 1)
            macro_calls = results_df['llm_macro_called'].sum()
            macro_success = results_df['llm_macro_success'].sum()
            macro_rate = (macro_success / macro_calls * 100) if macro_calls > 0 else 0
            print(f'   NIVEL 1 (Macro):')
            print(f'     Llamadas:  {macro_calls:,}')
            print(f'     Exitosas:  {macro_success:,} ({macro_rate:.1f}%)')
            
            # Category (Nivel 2)
            cat_calls = results_df['llm_category_called'].sum()
            cat_success = results_df['llm_category_success'].sum()
            cat_rate = (cat_success / cat_calls * 100) if cat_calls > 0 else 0
            print(f'   NIVEL 2 (Categoría):')
            print(f'     Llamadas:  {cat_calls:,}')
            print(f'     Exitosas:  {cat_success:,} ({cat_rate:.1f}%)')
            
            # Total
            total_success = macro_success + cat_success
            total_rate = (total_success / total_llm_calls * 100) if total_llm_calls > 0 else 0
            print(f'   TOTAL LLM:')
            print(f'     Llamadas:  {total_llm_calls:,}')
            print(f'     Exitosas:  {total_success:,} ({total_rate:.1f}%)')
            print(f'     Contribución: {total_success / results_df["auto_assigned"].sum() * 100:.1f}% del total auto-asignado')
    else:
        print('❌ NO hubo productos auto-asignados en ninguna iteración')
        print('   → Los thresholds (0.90/0.75) son DEMASIADO ALTOS para estos productos')
        print('   → Considerar bajar thresholds o usar estrategia manual de revisión')
else:
    print('⚠️  Solo 1 iteración - no hay suficiente data para graficar convergencia')